#  RAG 체인 구성
- Naïve RAG 구현

### **학습 목표:**  RAG 기반의 질의응답 시스템을 구현할 수 있다

### **실습 자료**: 

- data/transformer.pdf

---

# 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

`(3) 문서 로드`

In [3]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화
pdf_loader = PyPDFLoader('./data/transformer.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

PDF 문서 개수: 15


`(4) 텍스트 분할`

In [6]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# Hugging Face의 임베딩 모델 생성
embeddings_huggingface = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 토크나이저 직접 접근
tokenizer = embeddings_huggingface._client.tokenizer

# 토크나이저를 사용한 예시
text = "테스트 텍스트입니다."
tokens = tokenizer(text)
print(tokens)

# 토크나이저 설정 확인
print(tokenizer.model_max_length)  # 최대 토큰 길이
print(tokenizer.vocab_size)        # 어휘 크기

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

e:\study\modulab-ai\week1\faq_bot\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\guseh\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

{'input_ids': [0, 153924, 239355, 5826, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1]}
8192
250002


In [7]:
# 토큰 수를 계산하는 함수
def count_tokens(text):
    return len(tokenizer(text)['input_ids'])

# 토큰 수 계산
text = "테스트 텍스트입니다."
print(count_tokens(text))

6


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                      
    chunk_overlap=100,           
    length_function=count_tokens,         # 토큰 수를 기준으로 분할
    separators=["\n\n", "\n", " ", ""],   # 구분자 - 재귀적으로 순차적으로 적용 
)

# 텍스트 분할
chunks = text_splitter.split_documents(pdf_docs)
print(f"생성된 텍스트 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")
print(f"각 청크의 토큰 수: {list(count_tokens(chunk.page_content) for chunk in chunks)}")

생성된 텍스트 청크 수: 38
각 청크의 길이: [1378, 1796, 1831, 1857, 1292, 1609, 503, 1554, 1278, 1362, 1608, 833, 1418, 1680, 999, 1764, 1604, 539, 1219, 1645, 926, 1213, 1688, 716, 1409, 1626, 624, 1411, 1437, 913, 1493, 1337, 845, 812, 470, 438, 470, 441]
각 청크의 토큰 수: [336, 415, 405, 419, 327, 424, 127, 388, 294, 384, 411, 204, 419, 417, 226, 419, 395, 149, 390, 400, 221, 356, 411, 181, 394, 405, 188, 424, 399, 277, 420, 409, 250, 190, 131, 117, 131, 113]


In [9]:
# 청크의 텍스트 확인
print(chunks[2].page_content)

1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements in computational efficiency through facto

# 벡터 저장소 기반 RAG 검색기 (Retriever)



`(1) 벡터 저장소 초기화`
- chroma 사용
- cosine distance 기준으로 인덱싱 

In [10]:
from langchain_chroma import Chroma

# Chroma 벡터 저장소 생성하기
chroma_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_huggingface,    # huggingface 임베딩 사용
    collection_name="db_transformer",    # 컬렉션 이름
    persist_directory="./chroma_db",
    collection_metadata = {'hnsw:space': 'cosine'}, # l2, ip, cosine 중에서 선택 
)

# 현재 저장된 컬렉션 데이터 확인
chroma_db.get()

{'ids': ['c7dc21cb-bc97-4567-8290-062eb4db53bf',
  '2b993e7d-0436-4685-8ff9-23b794f27533',
  '06bcff00-5cfb-42d4-afc9-7fc1a81878d9',
  '9df65da5-6f27-43c1-ab04-7f872dc3beb0',
  '19b690d5-0afe-4978-94b9-2dd8016bb5a7',
  '26d7ffa3-bd91-4a5a-8ed7-2ea52de8ccec',
  'd310ff2d-ab7a-4b3b-adbf-b2f27c4825d2',
  '45dcea4d-8634-46b1-af89-94fb69b60b2b',
  'b8881634-cae7-4299-8feb-4c5ff230368d',
  '63ba9d6b-f60e-4422-9eb9-98c6ec3121c3',
  'e1593681-cf6a-4cfb-90ca-d9b678d15eb4',
  'e340f02e-96a9-4d25-8df7-2024ad9ad49b',
  '483a8bc6-5979-461e-9d2d-7e8e66e66877',
  '62596542-e5b1-4895-b9cf-05cac7f46267',
  'adb15d92-ab6d-40c1-89d2-da3f6d9462a5',
  '0bc7dbf4-4902-4b78-a18e-654878d2d518',
  'a6787b8c-9413-4e45-bc36-582c9fdf94e8',
  '7cad840c-990e-430a-803a-052d0e751156',
  '442660df-7072-4d31-bf6c-a08d8b1a9597',
  'b92883a6-4f66-4308-869c-c9a0b4f5027e',
  '88aa8f9d-7ae2-4b68-820b-9e4e1215d102',
  '19c2f97c-2220-4620-93b9-4ca209ba7bda',
  '10e11318-45f2-4ac0-85a6-4ee0fec5a91a',
  '9d85517b-d7c6-43a1-88f3-

`(2) Top K`

In [11]:
chroma_k_retriever = chroma_db.as_retriever(
    search_kwargs={"k": 2},
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_k_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [출처: {doc.metadata['source']}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural...he Transformer allows for significantly more parallelization and can reach a new state of the art in [출처: ./data/transformer.pdf]
----------------------------------------------------------------------------------------------------
-2-
In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley-
Parse... 2016.
[2] Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio. Neural machine translation by jointly [출처: ./data/transformer.pdf]
----------------------------------------------------------------------------------------------------


`(3) 임계값 지정`
- Similarity score threshold (기준 스코어 이상인 문서를 대상으로 추출)

In [12]:
from langchain_community.utils.math import cosine_similarity

chroma_threshold_retriever = chroma_db.as_retriever(
    search_type='similarity_score_threshold',       # cosine 유사도
    search_kwargs={'score_threshold': 0.5, 'k':2},  # 0.5 이상인 문서를 추출
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_threshold_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    score = cosine_similarity(
        [embeddings_huggingface.embed_query(query)], 
        [embeddings_huggingface.embed_query(doc.page_content)]
        )[0][0]
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [유사도: {score}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural...he Transformer allows for significantly more parallelization and can reach a new state of the art in [유사도: 0.5069071907792917]
----------------------------------------------------------------------------------------------------
-2-
In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley-
Parse... 2016.
[2] Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio. Neural machine translation by jointly [유사도: 0.5020666027962651]
----------------------------------------------------------------------------------------------------


`(4) MMR(Maximal Marginal Relevance) 검색`

In [13]:
# MMR - 다양성 고려 (lambda_mult 작을수록 더 다양하게 추출)
chroma_mmr = chroma_db.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 3,                 # 검색할 문서의 수
        'fetch_k': 8,           # mmr 알고리즘에 전달할 문서의 수 (fetch_k > k)
        'lambda_mult': 0.5,     # 다양성을 고려하는 정도 (1은 최소 다양성, 0은 최대 다양성을 의미. 기본값은 0.5)
        },
)


query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_mmr.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    score = cosine_similarity(
        [embeddings_huggingface.embed_query(query)], 
        [embeddings_huggingface.embed_query(doc.page_content)]
        )[0][0]
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]} [유사도: {score}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural...he Transformer allows for significantly more parallelization and can reach a new state of the art in [유사도: 0.5069071907792917]
----------------------------------------------------------------------------------------------------
-2-
Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for ...ng
corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We [유사도: 0.47915486221437187]
----------------------------------------------------------------------------------------------------
-3-
from our models and present and discuss examples in the appendix. Not only do individual attention
h..., according to the formula:
lrate = d−0.5
model · min(step_num−0.5, step_num · warmup_steps−1.5) (3) [유사도: 0.47091688357014927]
----------------------------------------------------------

`(5) metadata 필터링 검색`

In [14]:
# 메타데이터 확인
chunks[0].metadata

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2024-04-10T21:11:43+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2024-04-10T21:11:43+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': './data/transformer.pdf',
 'total_pages': 15,
 'page': 0,
 'page_label': '1'}

In [15]:
# 문서 객체의 metadata를 이용한 필터링
chrom_metadata = chroma_db.as_retriever(
    search_kwargs={
        'filter': {'source': './data/transformer.pdf'},
        'k': 5, 
        }
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chrom_metadata.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content} [출처: {doc.metadata['source']}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements i

`(6) page_content 본문 필터링 검색`

In [16]:
# page_content를 이용한 필터링
chroma_content = chroma_db.as_retriever(
    search_kwargs={
        'k': 2,
        'where_document': {'$contains': 'recurrent'},
        }
)

query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = chroma_content.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content} [출처: {doc.metadata['source']}]")
    print("-" * 100)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
-1-
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements i

# [실습 프로젝트] Naive RAG 구현 

- 각 단계별 지시사항에 따라 코드를 완성하세요. 
- 제시된 지시사항과 LangChain 문서를 참조하여 시스템을 구성합니다. 

`(1) 벡터 저장소 설정`
- HuggingFace에서 지원하는 BAAI/bge-m3 임베딩 모델을 사용하여 문서를 벡터화
- FAISS DB를 벡터 스토어로 사용 (IndexFlatL2 사용: 유클리드 거리)

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings  

# Hugging Face의 임베딩 모델 생성
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3-small")

# 임베딩 차원 확인
embedding = embeddings_model.embed_query("test")
print(f"임베딩 차원: {len(embedding)}")

임베딩 차원: 1024


In [ ]:
# Ollama 임베딩 모델을 사용한 FAISS 벡터 저장소 생성
import faiss 
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# FAISS 인덱스 초기화 (유클리드 거리 사용)
dim = len(embedding)  # 임베딩 차원
faiss_index = faiss.IndexFlatL2(dim)

# FAISS 벡터 저장소 생성
faiss_db = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,           # 벡터 검색을 위한 데이터 구조를 정의
    docstore=InMemoryDocstore(), # 문서 저장소 객체를 지정 - 문서의 원본 내용과 메타데이터를 보관
    index_to_docstore_id={},     # 인덱스와 문서 간의 연결을 관리 (매핑 딕셔너리)
)

# 저장된 문서의 갯수 확인
print(faiss_db.index.ntotal)

0


In [ ]:
import uuid
from langchain_core.documents import Document

documents = [
    ("인공지능은 컴퓨터 과학의 한 분야입니다.", "AI 개론"),
    ("머신러닝은 인공지능의 하위 분야입니다.", "AI 개론"),
    ("딥러닝은 머신러닝의 한 종류입니다.", "딥러닝 입문"),
    ("자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.", "AI 개론"),
    ("컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.", "딥러닝 입문")
]

doc_objects = []
for content, source in documents:
    doc = Document(
        page_content=content,
        metadata={"source": source},
    )
    doc_objects.append(doc)

# 문서 id 생성
doc_ids = [str(uuid.uuid4()) for _ in range(len(doc_objects))]

# 문서를 벡터 저장소에 저장
added_doc_ids = faiss_db.add_documents(documents=doc_objects, ids=doc_ids)

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(f"문서 IDs: {added_doc_ids}")
print(f"저장된 총 문서 개수: {faiss_db.index.ntotal}")

5개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['5a454e3e-5b83-4377-a52e-3da0f4e24eb9', 'cc143568-a9f8-41bc-ab14-b6f84f71d75b', '4d1625ec-8c7b-44d4-afce-3254c3b29328', '588771d1-b346-426c-95f0-623320f22ff6', '639d27fc-2481-450d-822f-00f3aa383651']


`(2) 검색기 정의`
- mmr 검색으로 상위 3개 문서 검색하는 Retriever 사용
- 다양성을 높이는 설정을 사용 

In [22]:
# mmr 검색기 생성
faiss_mmr_retriever = faiss_db.as_retriever(
    search_type="mmr",      # Maximum Marginal Relevance 검색 사용
    search_kwargs={
        "k": 3,             # 상위 3개 문서 검색
        "lambda_mult": 0.5, # 다양성 매개변수 (0에 가까울수록 다양성 높음, 1에 가까울수록 관련성 높음)
        "fetch_k": 10       # MMR에서 고려할 문서 개수 (더 많은 후보에서 선택)
    }
)

In [23]:
# 검색 테스트 
query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = faiss_mmr_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
print("=" * 80)
for i, doc in enumerate(retrieved_docs, 1):
    content = doc.page_content
    source = doc.metadata.get("source", "Unknown")
    
    # 내용이 100자 이하면 전체 출력, 초과하면 앞뒤 100자씩 출력
    if len(content) <= 100:
        display_content = content
    else:
        display_content = f"{content[:50]}...{content[-50:]}"
    
    print(f"[문서 {i}] 출처: {source}")
    print(f"내용: {display_content}")
    print("-" * 80)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
[문서 1] 출처: 딥러닝 입문
내용: 딥러닝은 머신러닝의 한 종류입니다.
--------------------------------------------------------------------------------
[문서 2] 출처: 딥러닝 입문
내용: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
--------------------------------------------------------------------------------
[문서 3] 출처: AI 개론
내용: 인공지능은 컴퓨터 과학의 한 분야입니다.
--------------------------------------------------------------------------------


`(3) RAG 프롬프트 구성`

- 작성 기준: 
    - LangChain의 ChatPromptTemplate 클래스 사용
    - 변수 처리는 {context}, {question} 형식 사용
    - 답변은 한글로 출력되도록 프롬프트 작성
    
- 아래 템플릿 코드를 기반으로 다음 내용을 참고하여 작성합니다. 

    1. 프롬프트 구성요소:
        - 작업 지침
        - 컨텍스트 영역
        - 질문 영역
        - 답변 형식 가이드

    2. 작업 지침:
        - 컨텍스트 기반 답변 원칙
        - 외부 지식 사용 제한
        - 불확실성 처리 방법
        - 답변 불가능한 경우의 처리 방법

    3. 답변 형식:
        - 핵심 답변 섹션
        - 근거 제시 섹션
        - 추가 설명 섹션 (필요시)

    4. 제약사항 반영:
        - 답변은 사실에 기반해야 함
        - 추측이나 가정을 최소화해야 함
        - 명확한 근거 제시가 필요함
        - 구조화된 형태로 작성되어야 함

In [52]:
# Prompt 템플릿 (예시)
from langchain.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context.

[Context]
{context}

[Question] 
{question}

[Answer]
"""

prompt = ChatPromptTemplate.from_template(template)

In [24]:
# Prompt 템플릿 (여기에 작성하세요)
from langchain.prompts import ChatPromptTemplate

template = """당신은 주어진 문서 컨텍스트를 기반으로 정확하고 유용한 답변을 제공하는 AI 어시스턴트입니다.

## 작업 지침
1. **컨텍스트 기반 답변 원칙**: 반드시 제공된 컨텍스트 정보만을 사용하여 답변하세요.
2. **외부 지식 사용 제한**: 컨텍스트에 없는 정보나 개인적인 지식을 추가하지 마세요.
3. **불확실성 처리**: 컨텍스트 정보가 불충분하거나 모호한 경우, 이를 명시하세요.
4. **답변 불가능한 경우**: 컨텍스트에서 답변할 수 없는 질문이라면, 정직하게 답변할 수 없다고 말하세요.

## 컨텍스트
{context}

## 질문
{question}

## 답변 형식
다음 구조에 따라 한글로 답변하세요:

**[핵심 답변]**
질문에 대한 직접적이고 명확한 답변을 제시하세요.

**[근거 제시]**
답변의 근거가 되는 컨텍스트 정보를 명시하세요.

**[추가 설명]** (필요시)
답변을 보완하는 관련 정보나 맥락을 제공하세요.

## 제약사항
- 답변은 반드시 사실에 기반해야 합니다
- 추측이나 가정을 최소화하세요
- 컨텍스트에서 명확한 근거를 찾을 수 있는 경우에만 답변하세요
- 구조화된 형태로 명확하게 작성하세요

답변:"""

prompt = ChatPromptTemplate.from_template(template)

# 템플릿 출력
prompt.pretty_print()

test_context = "\n".join([doc.page_content for doc in retrieved_docs])
test_question = "딥러닝과 머신러닝의 관계는 무엇인가요?"

formatted_prompt = prompt.format(
    context=test_context,
    question=test_question
)

print("포맷팅된 프롬프트:")
print("=" * 80)
print(formatted_prompt)
print("=" * 80)

================================ Human Message =================================

당신은 주어진 문서 컨텍스트를 기반으로 정확하고 유용한 답변을 제공하는 AI 어시스턴트입니다.

## 작업 지침
1. **컨텍스트 기반 답변 원칙**: 반드시 제공된 컨텍스트 정보만을 사용하여 답변하세요.
2. **외부 지식 사용 제한**: 컨텍스트에 없는 정보나 개인적인 지식을 추가하지 마세요.
3. **불확실성 처리**: 컨텍스트 정보가 불충분하거나 모호한 경우, 이를 명시하세요.
4. **답변 불가능한 경우**: 컨텍스트에서 답변할 수 없는 질문이라면, 정직하게 답변할 수 없다고 말하세요.

## 컨텍스트
{context}

## 질문
{question}

## 답변 형식
다음 구조에 따라 한글로 답변하세요:

**[핵심 답변]**
질문에 대한 직접적이고 명확한 답변을 제시하세요.

**[근거 제시]**
답변의 근거가 되는 컨텍스트 정보를 명시하세요.

**[추가 설명]** (필요시)
답변을 보완하는 관련 정보나 맥락을 제공하세요.

## 제약사항
- 답변은 반드시 사실에 기반해야 합니다
- 추측이나 가정을 최소화하세요
- 컨텍스트에서 명확한 근거를 찾을 수 있는 경우에만 답변하세요
- 구조화된 형태로 명확하게 작성하세요

답변:
포맷팅된 프롬프트:
Human: 당신은 주어진 문서 컨텍스트를 기반으로 정확하고 유용한 답변을 제공하는 AI 어시스턴트입니다.

## 작업 지침
1. **컨텍스트 기반 답변 원칙**: 반드시 제공된 컨텍스트 정보만을 사용하여 답변하세요.
2. **외부 지식 사용 제한**: 컨텍스트에 없는 정보나 개인적인 지식을 추가하지 마세요.
3. **불확실성 처리**: 컨텍스트 정보가 불충분하거나 모호한 경우, 이를 명시하세요.
4. **답변 불가능한 경우**: 컨텍스트에서 답변할 수 없는 질문이라면, 정직하게 답변할 수 없다고 말하세요.

## 컨텍스트
딥러닝은 머신러닝의 한 종류입니다.
컴퓨터

`(4) RAG 체인 구성`
- LangChain의 LCEL 문법을 사용
- 검색 결과를 프롬프트의 'context'로 전달하고,
- 사용자가 입력한 질문을 그래도 프롬프트의 'question'에 전달
- LLM 설정:
    - ChatOpenAI 사용 ('gpt-4.1-mini' 모델)
    - temperature: 답변의 일관성을 가져가는 설정값을 사용 
    - 기타 필요한 설정 
- 출력 파서: 문자열 부분만 출력되도록 구성 

In [25]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# LLM 설정
llm = ChatOpenAI(
    model="gpt-4o-mini",  # 정확한 모델명 사용
    temperature=0.1,      # 답변의 일관성을 위한 낮은 temperature 설정
    max_tokens=1000,      # 적절한 응답 길이 제한
    top_p=0.9,           # 응답의 다양성과 일관성 균형
    frequency_penalty=0.0, # 반복 억제
    presence_penalty=0.0   # 주제 일탈 방지
)


# 문서 포맷팅
def format_docs(docs):
    """검색된 문서들을 문자열로 포맷팅"""
    formatted_docs = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "Unknown")
        content = doc.page_content
        formatted_docs.append(f"[문서 {i}] 출처: {source}\n내용: {content}")
    return "\n\n".join(formatted_docs)

# RAG 체인 생성
rag_chain = (
    {
        "context": faiss_mmr_retriever | format_docs,  # 검색 결과를 포맷팅하여 context로 전달
        "question": RunnablePassthrough()              # 사용자 질문을 그대로 question으로 전달
    }
    | prompt        # 프롬프트 템플릿에 context와 question 삽입
    | llm          # LLM으로 응답 생성
    | StrOutputParser()  # 문자열 출력 파서로 결과 정리
)

# 체인 실행
query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
print(f"쿼리: {query}")
print("=" * 80)

try:
    output = rag_chain.invoke(query)
    print("답변:")
    print(output)
except Exception as e:
    print(f"오류 발생: {e}")
    print("OpenAI API 키가 설정되지 않았거나 모델에 접근할 수 없습니다.")
    print("실제 환경에서는 OpenAI API 키를 설정해야 합니다.")

print("=" * 80)


쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
답변:
**[핵심 답변]**  
대표적인 시퀀스 모델에 대한 정보는 제공된 컨텍스트에 포함되어 있지 않습니다.

**[근거 제시]**  
컨텍스트에는 딥러닝, 컴퓨터 비전, 인공지능에 대한 정보만 포함되어 있으며, 시퀀스 모델에 대한 언급은 없습니다.

**[추가 설명]**  
따라서, 시퀀스 모델에 대한 구체적인 답변을 제공할 수 없습니다.


`(5) Gradio 스트리밍 구현`
- ChatInterface 사용
- `chain.stream()`으로 응답을 청크 단위로 스트리밍

In [28]:
import gradio as gr
from typing import Iterator

# 스트리밍 응답 생성 함수
def get_streaming_response(message: str, history) -> Iterator[str]:
    """
    RAG 체인을 사용하여 스트리밍 응답을 생성하는 함수
    
    Args:
        message (str): 사용자 입력 메시지
        history: 채팅 히스토리 (ChatInterface에서 자동으로 관리)
    
    Yields:
        str: 누적된 응답 텍스트
    """
    try:
        # RAG Chain 실행 및 스트리밍 응답 생성
        response = ""
        
        # rag_chain.stream()을 사용하여 청크 단위로 응답 받기
        for chunk in rag_chain.stream(message):
            if isinstance(chunk, str):
                response += chunk
                yield response
            elif hasattr(chunk, 'content'):  # AIMessage 객체인 경우
                response += chunk.content
                yield response
                
    except Exception as e:
        # 에러 발생 시 사용자에게 알림
        error_message = f"⚠️ 응답 생성 중 오류가 발생했습니다: {str(e)}\n\n"
        error_message += "OpenAI API 키가 설정되어 있는지 확인해주세요."
        yield error_message


# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=get_streaming_response,
    title="🤖 RAG 기반 AI 챗봇",
    description="...",
    examples=[
        "인공지능이란 무엇인가요?",
        "머신러닝과 딥러닝의 차이점은 무엇인가요?",
        "자연어 처리 기술에 대해 설명해주세요.",
        "컴퓨터 비전은 어떤 분야인가요?",
        "딥러닝이 머신러닝의 하위 분야인 이유는?"
    ],
    theme=gr.themes.Soft()
)

# 실행
def launch_demo(share=False, debug=False):
    """
    Gradio 데모를 실행하는 함수
    
    Args:
        share (bool): 공개 링크 생성 여부
        debug (bool): 디버그 모드 실행 여부
    """
    print("\n=== Gradio RAG 챗봇 실행 ===")
    print("🚀 RAG 기반 AI 챗봇이 시작됩니다...")
    print("📚 지식 베이스: AI/ML/DL 관련 문서")
    print("🔍 검색 방식: FAISS + MMR")
    print("🤖 모델: GPT-4o-mini")
    
    if not share:
        print("\n💡 참고: share=True로 설정하면 공개 링크를 생성할 수 있습니다.")
    
    try:
        demo.launch(
            share=share,
            debug=debug,
            show_error=True,        # 에러 표시
            quiet=False            # 로그 출력
        )
    except Exception as e:
        print(f"❌ Gradio 실행 중 오류 발생: {e}")
        print("포트가 이미 사용 중이거나 다른 문제가 있을 수 있습니다.")


if __name__ == "__main__":
    launch_demo(share=False, debug=True)

e:\study\modulab-ai\week1\faq_bot\.venv\Lib\site-packages\gradio\chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(



=== Gradio RAG 챗봇 실행 ===
🚀 RAG 기반 AI 챗봇이 시작됩니다...
📚 지식 베이스: AI/ML/DL 관련 문서
🔍 검색 방식: FAISS + MMR
🤖 모델: GPT-4o-mini

💡 참고: share=True로 설정하면 공개 링크를 생성할 수 있습니다.
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


In [ ]:
# demo 실행 종료
demo.close()